## XML to relational table

Extract colonist data into two normalized DataFrames:
- **pawns_df** — one row per colonist (`pawn_id`, `first`, `last`)
- **skills_df** — one row per skill entry (`pawn_id`, `skill`, `level`)

In [17]:
import xml.etree.ElementTree as ET
import pandas as pd

tree = ET.parse("logs/09_04_2026.rws")
root = tree.getroot()

### 1. Build the `pawns` table
- One row per colonist.
- Columns: `pawn_id` (auto-assigned integer), `first`, `last`.

- Use the same XPath from exercise 11 to get the colonist list
- `pawn_id` must be assigned manually — there's a built-in that gives you both the index and value while iterating
- use the method that safely returns `None` when the tag is missing (instead of `.find().text`)
- Build a dict per row, collect them in a list, then pass to `pd.DataFrame()`

In [18]:
things = root.find("game/maps/li/things")

pawn_list = []

pawns = things.findall(".//thing[@Class='Pawn'][kindDef='Colonist']")
for pawn_id, pawn in enumerate(pawns):
    pawn_list.append({
        'pawn_id': pawn_id,
        'first': pawn.findtext('name/first'),
        'last': pawn.findtext('name/last')
    })

pawns_df = pd.DataFrame(pawn_list)

print(pawns_df)

    pawn_id first   last
0         0    티코   크레이그
1         1   나나미  커즈이스크
2         2   앤소니   크레이그
3         3    콜린  하이드로넷
4         4   다프네  하이드로넷
5         5  루드밀라     피셔
6         6   히스인   라우이트
7         7   헤스크  커즈이스크
8         8  베네딕트     노턴
9         9   마나부  하이드로넷
10       10  히스이스   윈'하욘
11       11   마룰로  하이드로넷
12       12   피노보    호네본
13       13   이리나   크레이그
14       14  그레이시   플로레스
15       15  그레고리  하이드로넷


### 2. Build the `skills` table
- One row per skill per colonist.
- Columns: `pawn_id`, `skill`, `level`.
- Skills omitted from XML mean level 0 — fill them in explicitly.
- Build a `{skill_name: level}` dict from each pawn's XML entries — it makes filling in missing skills straightforward
- Assign `pawn_id` the same way as in exercise 1 so the two tables can be `merge` later

In [19]:
ALL_SKILLS = [
    "Shooting", "Melee", "Construction", "Mining", "Cooking", 
    "Plants", "Animals", "Crafting", "Artistic", "Medicine",
    "Social", "Intellectual"
]

things = root.find("game/maps/li/things")
pawns = things.findall(".//thing[@Class='Pawn'][kindDef='Colonist']")

skill_list = []

for pawn_id, pawn in enumerate(pawns):
    skills = {
        skill.findtext('def'): skill.findtext('level') or 0
        for skill in pawn.findall("skills/skills/li")
    }

    for skill_name in ALL_SKILLS:
        skill_list.append({
            'pawn_id': pawn_id,
            'skill': skill_name,
            'level': skills.get(skill_name, 0)
        })

skills_df = pd.DataFrame(skill_list)
print(skills_df)

     pawn_id         skill level
0          0      Shooting     8
1          0         Melee     1
2          0  Construction     1
3          0        Mining     6
4          0       Cooking     8
..       ...           ...   ...
187       15      Crafting     0
188       15      Artistic     0
189       15      Medicine     0
190       15        Social     0
191       15  Intellectual     0

[192 rows x 3 columns]


### 3. Merge two tables
- Merge two tables(pawn list and skill list) based on `pawn_id`

In [20]:
merged_df = pd.merge(pawns_df, skills_df, on='pawn_id', how='inner')

print(merged_df)

     pawn_id first   last         skill level
0          0    티코   크레이그      Shooting     8
1          0    티코   크레이그         Melee     1
2          0    티코   크레이그  Construction     1
3          0    티코   크레이그        Mining     6
4          0    티코   크레이그       Cooking     8
..       ...   ...    ...           ...   ...
187       15  그레고리  하이드로넷      Crafting     0
188       15  그레고리  하이드로넷      Artistic     0
189       15  그레고리  하이드로넷      Medicine     0
190       15  그레고리  하이드로넷        Social     0
191       15  그레고리  하이드로넷  Intellectual     0

[192 rows x 5 columns]


### 4. Build a pivot table
- Reshape `merged_df` so each row is a colonist and each column is a skill.
- The values should be the skill `level`.
- Use `pd.pivot_table()` or the DataFrame `.pivot()` method
- Use `first` (first name) as the row index

In [24]:
pivot_df = merged_df.pivot(index='first', columns='skill', values='level')
pivot_df

skill,Animals,Artistic,Construction,Cooking,Crafting,Intellectual,Medicine,Melee,Mining,Plants,Shooting,Social
first,,,,,,,,,,,,
그레고리,0,0,0,0,0,0,0,0,0,0,0,0
그레이시,11,1,2,3,1,2,2,3,6,15,13,13
나나미,2,6,15,12,13,2,0,8,12,11,4,2
다프네,1,1,14,5,3,1,3,8,4,3,7,6
루드밀라,10,1,1,14,2,1,11,2,5,11,0,8
마나부,5,0,1,14,4,0,2,7,14,8,0,4
마룰로,5,1,2,3,0,4,3,13,2,14,5,1
베네딕트,2,0,0,2,11,1,7,8,1,2,4,7
앤소니,5,0,1,12,1,1,0,4,1,3,11,9


### 5. Draw a skill heatmap
- Visualize the pivot table as a heatmap using `matplotlib`.
- Hints:
  - `import matplotlib.pyplot as plt`
  - Use `plt.imshow()` to render the 2D array — pass the pivot table's `.values`
  - Set the X-axis ticks to skill names (`pivot_df.columns`) and Y-axis ticks to colonist names (`pivot_df.index`)
  - Add a colorbar with `plt.colorbar()` so the level scale is visible
  - Rotate X-axis labels 45° for readability: `plt.xticks(rotation=45, ha='right')`
  - Give the plot a title and call `plt.tight_layout()` before `plt.show()`

In [22]:
# Your code here


### 6. Find the best colonist per skill
- For each skill, find the colonist with the highest level.
- Print the result as a clean table with columns: `skill`, `best_colonist`, `level`.
- Hints:
  - Work from `merged_df`, not the pivot table
  - `groupby` + `idxmax()` is one approach: find the row index of the max level per skill, then use `.loc[]` to retrieve the full row
  - Filter out skills where every colonist is level 0 (no one has that skill)
  - Sort the final result by `level` descending

In [23]:
# Your code here
